In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Connect an agent to store tools over MCP

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Model Context Protocol (MCP)

The [Model Context Protocol](https://modelcontextprotocol.io/) is an open standard for serving tools to AI applications. A team publishes its tools once on an MCP server, and any MCP client can discover and call them without importing the team's code.

### MCP tools in ADK

In ADK, [`McpToolset`](https://adk.dev/mcp/) is the client. It connects to a server when the agent first needs its tools, lists them, and gives them to the model like any other tool. The connection can be a local process over stdio (this quickstart) or a remote server over HTTP (the deployed store agent).

### Store scope stays with the agent

An MCP server has no session, so it cannot know who is signed in. The agent that owns the toolset checks the store scope in a `before_tool_callback`, then fills in the signed-in store as the `store_id` argument. The model never has to type the store id, and a request for another store is refused before anything leaves the agent.

<img width="60%" src="../../docs/diagrams/q08.png" alt="The agent calls store tools on an MCP server through McpToolset, with a scope check before each call" />

### Objectives

In this tutorial, you will learn how to give an ADK agent tools that are served over MCP.

You will complete the following tasks:

- Start the store MCP server as a local process and list its tools
- Build an agent that uses those tools, with a scope check before each call
- Ask stock questions and see a request for another store refused
- See how the same agent connects to a remote MCP server

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded in the workshop notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The quickstart imports shared store code from the repository root, two folders up
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agents" / "cymbal_store_ops").is_dir())
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the output to the agent's own events: ADK marks experimental and deprecated features
# with warnings and logs configuration hints, and the Gen AI SDK logs a note whenever a
# response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_adk").setLevel(logging.ERROR)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [4]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import BaseTool, ToolContext
from google.adk.tools.mcp_tool import McpTool, McpToolset, StdioConnectionParams
from google.genai import types
from mcp import StdioServerParameters

from agents.cymbal_store_ops.callbacks import enforce_store_scope_before_tool

## Connect to the MCP server

### The server

`cymbal_mcp_server.py` in this folder is a small MCP server built with the MCP Python SDK's `FastMCP`. It serves two read-only tools, `get_osa_exceptions` and `check_store_stock`, which call the same store functions the main agent uses. Each tool is one decorated function:

```python
mcp = FastMCP("cymbal-beauty-store-ops")

@mcp.tool(description=..., annotations=ToolAnnotations(readOnlyHint=True))
def check_store_stock(product_name: str, city: str = "", store_id: str = "") -> dict:
    return bad_store(store_id, city) or domain_tools.check_store_stock(
        product_name=product_name, city=city, store_id=store_id
    )
```

### Start it and list its tools

`StdioServerParameters` says how to start the server: the Python interpreter, the script, and the environment it needs. A stdio server inherits only a few basic variables, so pass it the project, the namespace and the path to the shared store code.

In [5]:
server = StdioServerParameters(
    command=sys.executable,
    args=["cymbal_mcp_server.py"],
    env={
        "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
        "WORKSHOP_NAMESPACE": WORKSHOP_NAMESPACE,
        "STORE_OPS_ENV": "dev",
        "PYTHONPATH": str(REPO_ROOT),
    },
)

toolset = McpToolset(connection_params=StdioConnectionParams(server_params=server, timeout=30))

`get_tools()` starts the server, opens an MCP session and asks which tools it serves. The agent code never imports the server; everything below comes over the protocol:

In [6]:
tools = await toolset.get_tools()

for tool in tools:
    print(f"{tool.name}\n  {tool.description.splitlines()[0]}\n")

check_store_stock
  Stock position of one product: on_hand, on_shelf_qty, backroom_qty, reorder_point, shelf_capacity, BOPIS

get_osa_exceptions
  List the store's on-shelf-availability exceptions with a recommendation for each.



## Build the agent

### Fill in the signed-in store

This callback runs before every tool call. For an MCP tool with no `store_id`, it adds the store from the session state. It runs after `enforce_store_scope_before_tool`, which has already refused a session with nobody signed in, or a request for a store the caller may not see.

In [7]:
def stamp_signed_in_store(tool: BaseTool, args: dict, tool_context: ToolContext) -> dict | None:
    """Give an MCP call without a store_id the signed-in store."""
    if isinstance(tool, McpTool) and not args.get("store_id"):
        args["store_id"] = tool_context.state["user:store_id"]
    return None

### Define the agent

The toolset goes in `tools` like any other tool. The two callbacks run in order: the scope check first, then the stamp.

In [8]:
agent = LlmAgent(
    name="mcp_tools_agent",
    model="gemini-3.8-flash",
    instruction="""You help Cymbal Beauty store managers. Your store tools arrive over MCP.
Signed-in store: {user:store_id?}, role: {user:role?}.
Leave store_id empty: the signed-in store is filled in for you. Answer in two or three
sentences, only from tool results, with the product id and the recommendation. If a tool
returns status ERROR, say what could not be done.""",
    tools=[toolset],
    before_tool_callback=[enforce_store_scope_before_tool, stamp_signed_in_store],
)

## Run the agent

Start a session as Dana, the manager of store S-014, and define a helper that prints each tool call and the answer.

In [9]:
runner = InMemoryRunner(agent=agent, app_name="mcp_tools_agent")
session = await runner.session_service.create_session(
    app_name="mcp_tools_agent",
    user_id="dana",
    state={
        "user:user_id": "U-M014",
        "user:store_id": "S-014",
        "user:role": "store_manager",
        "user:first_name": "Dana",
    },
)

In [10]:
async def ask(question: str) -> None:
    """Send one message and print the tool calls, transfers and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            if call.name == "transfer_to_agent":
                print(f"[{event.author}] hands over to {call.args.get('agent_name')}")
            else:
                print(f"[{event.author}] calls {call.name}({dict(call.args or {})})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n[{event.author}] {text}\n")

### Ask about stock

The model calls `check_store_stock` without a `store_id`; the callback adds `S-014` before the request goes to the server.

In [11]:
await ask("How much Lumière Hydra Cream do we have, and why is it flagged?")

[mcp_tools_agent] calls check_store_stock({'product_name': 'Lumière Hydra Cream'})



[mcp_tools_agent] We have 7 units of Lumière Hydra Cream (product ID P-0101) on hand, with 0 on the shelf and all 7 sitting in the backroom. It is flagged because the shelf is empty while stock is in the backroom, and the total inventory is below the reorder point of 12 with delayed replenishment. The system's recommendation is backroom_check.



Look for the stock position from the server: 7 units on hand, 0 on the shelf and all 7 in the backroom, below the reorder point of 12, with the recommendation `backroom_check`. The wording differs from run to run; the tool call and the numbers should match.

In [12]:
await ask("What are my top three on-shelf availability exceptions right now?")

[mcp_tools_agent] calls get_osa_exceptions({'limit': 3})



[mcp_tools_agent] Your top three on-shelf availability exceptions are Lumière Hydra Cream (product ID P-0101), Bloom Concealer (product ID P-0548), and Glow Blush (product ID P-0491). All three products currently have zero units on the shelf despite having available stock in the backroom. The recommendation for each of these items is backroom_check.



`get_osa_exceptions` also ran without a `store_id`. The three exceptions are P-0101, P-0548 and P-0491, each an empty shelf with stock in the backroom, and each with the recommendation `backroom_check`.

### Ask about another store

Dana manages S-014, so a request for S-020 is refused by the scope check. The model still asks for the call, but nothing is sent to the MCP server.

In [13]:
await ask("Show me the on-shelf availability exceptions for store S-020.")

[mcp_tools_agent] calls get_osa_exceptions({'store_id': 'S-020'})



[mcp_tools_agent] The on-shelf availability exceptions for store S-020 could not be retrieved because the request was blocked by policy, as this session is scoped to store S-014. With your store manager role restricted to store S-014, exceptions and recommendations for other stores cannot be accessed.



The model asked for `store_id: 'S-020'`, so the call line is printed, but the scope check answered it with an error before the MCP server was called. The agent then says what could not be done.

## Connect to a remote MCP server

The deployed store agent's production path is this same pattern: it reads every store record through the store MCP service on Cloud Run (`services/store_mcp/server.py`, 30 tools, wired in [`agents/cymbal_store_ops/mcp_catalog.py`](../../agents/cymbal_store_ops/mcp_catalog.py)). Only the connection changes: a URL instead of a command, and a header provider that adds the caller's identity token and the signed store scope to each request. You can see the full version in [`agents/cymbal_store_ops/mcp_connection.py`](../../agents/cymbal_store_ops/mcp_connection.py).

```python
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(url=f"{MCP_URL}/mcp", timeout=30),
    header_provider=make_header_provider(settings, caller),
)
```

The agent, its instruction and its callbacks stay the same.

## Cleaning up

Close the toolset to stop the local MCP server process.

In [14]:
await toolset.close()

## What's next

- [MCP tools in ADK](https://adk.dev/mcp/)
- [The store MCP service](../../docs/patterns/mcp-service.md): the deployed server, its authentication and its registration in Agent Registry
- [Quickstart guide](README.md) for running this agent in the ADK developer UI
- Next quickstart: [Guardrails](../09-guardrails-agent/walkthrough.ipynb)